# Les retrievers

In [31]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader

## Initialisation de modèle

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\mbial\AppData\Local\Temp\ipykernel_21308\412152783.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [39]:
llm =  ChatOpenAI(
    model=  "gpt-4.1-mini",
    temperature=1.0,
    #model_kwargs = {"top_p":0.2, "top_k":1},
    max_tokens=256,
    api_key=os.getenv("OPENAI_API_KEY")
)

## Découpage du texte

In [2]:
def text_splitter(chunk_size:int=1000, chunk_overlap:int=200):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

In [9]:
text_splitter = text_splitter()

In [8]:
data = TextLoader("text.txt").load()

In [13]:
chunks = text_splitter.split_documents(data)

## Base de données vectorielle

In [14]:
vector_db =  Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="my_text_collection"   
)

## Recherche simple

In [ ]:
retriever = vector_db.as_retriever()

In [16]:
query = "Email policy"

In [17]:
docs = retriever.invoke(query)

In [18]:
docs

[Document(metadata={'source': 'text.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'text.txt'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn't interfere with work responsibilities.\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\nConfidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensi

In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'text.txt'}, page_content='3.\tInternet and Email Policy')]

## La recherche MMR

In [22]:
retriever = vector_db.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'text.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'text.txt'}, page_content='2.\tRecruitment Policy'),
 Document(metadata={'source': 'text.txt'}, page_content='5.\tSmoking Policy'),
 Document(metadata={'source': 'text.txt'}, page_content='The Anti-Discrimination and Harassment Policy is a testament to the commitment of this organization in fostering a workplace that is free from discrimination, harassment, and any form of unlawful bias. This policy applies to every individual within the organization, including employees, contractors, visitors, and clients.\nNon-Discrimination: This organization strictly prohibits discrimination based on race, color, religion, gender, national origin, age, disability, sexual orientation, or any other legally protected characteristic in all aspects of employment, including recruitment, hiring, compensation, benefits, promotions, and terminations.\nHarassment: Harassment in any form, wh

## La recherche suivant un score de similarité

In [23]:
retriever = vector_db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4}
)
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'text.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'text.txt'}, page_content="Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and the importance of adhering to principles that maintain security, productivity, and legal compliance.\nAcceptable Use: Company-provided internet and email services are primarily meant for job-related tasks. Limited personal use is allowed during non-work hours, provided it doesn't interfere with work responsibilities.\nSecurity: Safeguard your login credentials, avoiding the sharing of passwords. Exercise caution with email attachments and links from unknown sources. Promptly report any unusual online activity or potential security breaches.\nConfidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensi

## <multi-query retriever>

## Multi-query retriever

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NCZCJ26bp3uKTa0gp8Agwg/multiquery.png" width="40%" alt="multiquery"/>


In [34]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.retrievers.multi_query import MultiQueryRetriever

In [25]:
loader = PyPDFLoader("Guide-Visiteur-CAN2021.pdf")
pdf_data = loader.load()

In [28]:
chunks_pdf = text_splitter.split_documents(pdf_data)

In [29]:
ids = vector_db.get()["ids"]
vector_db.delete(ids) # We need to delete existing embeddings from previous documents and then store current document embeddings in.
vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=embedding_model, collection_name="my_pdf_collection")

In [35]:
query = "Que visiter à Douala ?"

In [41]:
retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm
)

In [42]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [43]:
docs = retriever.invoke(query)
docs

INFO:langchain.retrievers.multi_query:Generated queries: ['Quels sont les principaux sites touristiques à découvrir à Douala ?  ', 'Quelles activités culturelles et lieux d’intérêt peut-on visiter à Douala ?  ', 'Quels endroits incontournables visiter lors d’un séjour à Douala ?']


[Document(metadata={'trapped': '/False', 'page_label': '71', 'moddate': '2022-01-28T18:52:54+01:00', 'creator': 'Adobe InDesign CS5 (7.0)', 'producer': 'Adobe PDF Library 9.9', 'source': 'Guide-Visiteur-CAN2021.pdf', 'creationdate': '2022-01-28T18:46:48+01:00', 'page': 70, 'total_pages': 110}, page_content='Mot de bienvenue du Maire \nde la ville de Douala\nMesdames et Messieurs les officiels, acteurs et supporteurs\nC’ est un insigne honneur et une réelle satisfaction \npour la ville de Douala dont je suis le 1er Magistrat \nmunicipal, ainsi que ses populations, d’ avoir été choisie \npour abriter une des poules de la 33è édition de la Coupe \nd’ Afrique des Nations de football au Cameroun.\nDe par sa situation géographique hautement stratégique, \nDouala est non seulement la capitale économique \ndu Cameroun, mais aussi et surtout la grande porte \nd’ entrée des pays de la sous-région CEMAC.\nA cet égard, hormis ses nombreuses opportunités \nd’ affaires, Douala et ses environs consti

## Parent Document retriever

In [44]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain.storage import InMemoryStore

In [45]:
parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [46]:
store = InMemoryStore()

In [50]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [52]:
retriever.add_documents(pdf_data)

In [53]:
len(list(store.yield_keys()))

155

In [54]:
sub_docs = vectordb.similarity_search("Visiter Douala")
print(sub_docs[0].page_content)

Poule de Douala, 
Région du Littoral
1. Douala en bref
2. Informations pratiques et 
utiles pour votre séjour
3. Que découvrir dans la 
ville et ses environs…
Monument Rond-point Deido à Douala (Frank Bofeli Jovago)


In [55]:
retrieved_docs = retriever.invoke("Visiter Douala")
print(retrieved_docs[0].page_content)

lE gUIDE DU vISITEUR
50
1. Ce qu’il faut savoir avant 
de se rendre à Douala
1.1. Douala en raccourci
Douala est la principale porte d’ en-
trée du Cameroun, par sa proximité 
au fleuve Wouri qui se jette directe-
ment dans la mer. Cette proximité 
à l’Océan Atlantique va favoriser le 
commerce des esclaves, la traite, et 
plus tard  l’implantation des du com-
merce de l’ivoire, huile de palme.
Les  Portugais, qui sont les premiers 
Européens à visiter l’ espace occupé 
par les Duala donnent au fleuve 
Wouri le nom de Rio dos Cameroes, 
qui par altération devient Came-
roons, puis Kamerun et enfin Came-
roun. Les premiers missionnaires qui 
ont débarqué dans cette ville sont 
les baptistes,  avec Joseph Merrick, 
Jackson Fuller, Alfred Saker qui vont 
s’installer et s’ engager dans l’ œuvre 
missionnaire des peuples côtiers avec 
la construction des temples, la tra-
duction de la bible en duala, bassa, 
et se sont également engagés dans 
la formation des pasteurs. Après les
